In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.utils as vutils
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np


c:\Users\jesli\anaconda3\envs\pv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [4]:
# Adjust these based on your GPU memory
gpu_memory = 8  # Set to 8 or 24 depending on your GPU

if gpu_memory == 8:
    batch_size = 32
    image_size = 128
    grad_accum_steps = 1  # Increase if necessary
elif gpu_memory == 24:
    batch_size = 128
    image_size = 256
    grad_accum_steps = 1

# Common hyperparameters
num_epochs = 100
lr = 0.0002
beta1 = 0.5
beta2 = 0.999
nz = 100  # Size of z latent vector (noise)
ngf = 64  # Size of feature maps in generator
ndf = 64  # Size of feature maps in discriminator
num_workers = os.cpu_count()


In [5]:
class PestDataset(torch.utils.data.Dataset):
    def __init__(self, image_folder, transform=None):
        self.image_paths = [os.path.join(image_folder, img) for img in os.listdir(image_folder)]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

# Data transformations
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)  # Normalize between -1 and 1
])

# Dataset and DataLoader
dataset = PestDataset(image_folder='gan_data', transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                        pin_memory=True)


In [6]:
class Generator(nn.Module):
    def __init__(self, nz, ngf, nc=3):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # Input is Z, going into a convolution
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # State size: (ngf*8) x 4 x 4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # State size: (ngf*4) x 8 x 8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # State size: (ngf*2) x 16 x 16
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # State size: (ngf) x 32 x 32
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
            # Output size: (nc) x 64 x 64
        )
        # Adjust final layer if image_size is larger
        if image_size == 128:
            self.main.add_module('extra_convtranspose', nn.ConvTranspose2d(nc, nc, 4, 2, 1, bias=False))
            self.main.add_module('extra_tanh', nn.Tanh())
        elif image_size == 256:
            for _ in range(2):
                self.main.add_module('extra_convtranspose', nn.ConvTranspose2d(nc, nc, 4, 2, 1, bias=False))
                self.main.add_module('extra_tanh', nn.Tanh())

    def forward(self, input):
        return self.main(input)


In [7]:
class Discriminator(nn.Module):
    def __init__(self, ndf, nc=3):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # Input is (nc) x image_size x image_size
            nn.Conv2d(nc, ndf, 3, 1, 1, bias=False),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(ndf, ndf*2, 3, 1, 1, bias=False),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(ndf*2, ndf*4, 3, 1, 1, bias=False),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(ndf*4, ndf*8, 3, 1, 1, bias=False),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),
        )

        # Output layer
        self.fc = nn.Sequential(
            nn.Linear((ndf*8)*(image_size//16)*(image_size//16), 1),
            nn.Sigmoid()
        )

    def forward(self, input):
        features = self.main(input)
        features = features.view(features.size(0), -1)
        output = self.fc(features)
        return output


In [8]:
# Create the generator and discriminator
netG = Generator(nz, ngf).to(device)
netD = Discriminator(ndf).to(device)

# Initialize weights
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1 or classname.find('InstanceNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG.apply(weights_init)
netD.apply(weights_init)

# Loss function
criterion = nn.BCELoss()

# Optimizers
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, beta2))


In [9]:
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"Current Device: {torch.cuda.current_device()}")
print(f"Device Name: {torch.cuda.get_device_name(torch.cuda.current_device())}")
netG = netG.to(device)
netD = netD.to(device)
real_cpu = real_cpu.to(device)


GPU Available: True
Current Device: 0
Device Name: NVIDIA GeForce RTX 4060 Laptop GPU


NameError: name 'real_cpu' is not defined

In [9]:
real_label = 1.0
fake_label = 0.0

G_losses = []
D_losses = []

for epoch in range(num_epochs):
    for i, data in enumerate(dataloader):
        ############################
        # (1) Update D network
        ###########################
        netD.zero_grad()
        # Format batch
        real_cpu = data.to(device, non_blocking=True)
        b_size = real_cpu.size(0)
        label = torch.full((b_size,), real_label, dtype=torch.float, device=device)
        output = netD(real_cpu).view(-1)
        errD_real = criterion(output, label)
        errD_real.backward()
        D_x = output.mean().item()

        # Generate fake images
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake = netG(noise)
        label.fill_(fake_label)
        output = netD(fake.detach()).view(-1)
        errD_fake = criterion(output, label)
        errD_fake.backward()
        D_G_z1 = output.mean().item()

        errD = errD_real + errD_fake
        optimizerD.step()

        ############################
        # (2) Update G network
        ###########################
        netG.zero_grad()
        label.fill_(real_label)  # Generator wants discriminator to think images are real
        output = netD(fake).view(-1)
        errG = criterion(output, label)
        errG.backward()
        D_G_z2 = output.mean().item()
        optimizerG.step()

        # Output training stats
        if i % 50 == 0:
            print(f'[{epoch}/{num_epochs}][{i}/{len(dataloader)}] '
                  f'Loss_D: {errD.item():.4f} Loss_G: {errG.item():.4f} '
                  f'D(x): {D_x:.4f} D(G(z)): {D_G_z1:.4f}/{D_G_z2:.4f}')

        # Save Losses for plotting later
        G_losses.append(errG.item())
        D_losses.append(errD.item())

    # Save generated images after every epoch
    with torch.no_grad():
        fake = netG(torch.randn(64, nz, 1, 1, device=device)).detach().cpu()
    vutils.save_image(fake, f'generated_epoch_{epoch}.png', normalize=True)

    # Save model weights every 10 epochs
    if epoch % 10 == 0:
        torch.save(netG.state_dict(), f"generator_epoch_{epoch}.pth")
        torch.save(netD.state_dict(), f"discriminator_epoch_{epoch}.pth")


In [ ]:
# Modify training loop with gradient accumulation
accumulation_steps = grad_accum_steps

for epoch in range(num_epochs):
    optimizerD.zero_grad()
    optimizerG.zero_grad()
    for i, data in enumerate(dataloader):
        # Rest of the training loop remains the same, but adjust optimizer steps
        # Accumulate gradients
        if (i + 1) % accumulation_steps == 0:
            optimizerD.step()
            optimizerG.step()
            optimizerD.zero_grad()
            optimizerG.zero_grad()


In [3]:
print(torch.cuda.memory_summary(device=device, abbreviated=False))


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [ ]:
plt.figure(figsize=(10,5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(G_losses, label="G")
plt.plot(D_losses, label="D")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
# Initialize models
netG = Generator(nz, ngf).to(device)
netD = Discriminator(ndf).to(device)

# Load state dictionaries
netG.load_state_dict(torch.load('generator_epoch_X.pth'))
netD.load_state_dict(torch.load('discriminator_epoch_X.pth'))


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.cuda.amp import GradScaler, autocast
from torchvision.utils import save_image
import os
import numpy as np
from tqdm import tqdm


# Hyperparameters
latent_dim = 100
img_size = 300
channels = 3
batch_size = 64
lr = 0.0002
b1 = 0.5
b2 = 0.999
n_epochs = 250
sample_interval = 500

img_shape = (channels, img_size, img_size)

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()

        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.Dropout(0.25), 
            
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(128, channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        z = z.view(z.size(0), latent_dim, 1, 1)
        img = self.model(z)
        return img

# Updated Discriminator to resemble YOLO initial layers
class YOLOLikeDiscriminator(nn.Module):
    def __init__(self):
        super(YOLOLikeDiscriminator, self).__init__()

        # YOLO-like convolutional layers
        self.model = nn.Sequential(
            # Layer 1: Similar to YOLO's first layer
            nn.Conv2d(channels, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1, inplace=True),
            
            # Layer 2: YOLO-style downsampling (like MaxPool)
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1, inplace=True),
            
            # Layer 3: Another downsampling and convolution layer like YOLO
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1, inplace=True),
            
            # Layer 4: More feature extraction, with larger filters
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1, inplace=True),
            
            # Layer 5: YOLO-inspired larger layer to expand receptive field
            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1, inplace=True),

            # Global Average Pooling to reduce feature maps to 1x1
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # Output layer for binary classification
        self.output_layer = nn.Sequential(
            nn.Conv2d(512, 1, kernel_size=1, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img):
        features = self.model(img)
        validity = self.output_layer(features)
        return validity.view(validity.size(0), -1)  # Flatten to (batch_size, 1)

# Initialize generator and YOLO-like discriminator
generator = Generator()
discriminator = YOLOLikeDiscriminator()

# Loss function
adversarial_loss = nn.BCELoss()

# Optimizers
optimizer_G = torch.optim.Adam(generator.parameters(), lr=lr, betas=(b1, b2))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(b1, b2))

num_gen_params = sum(p.numel() for p in generator.parameters())
num_disc_params = sum(p.numel() for p in discriminator.parameters())

print(f"Number of parameters in the generator: {num_gen_params}")
print(f"Number of parameters in the YOLO-like discriminator: {num_disc_params}")

from PIL import Image
from torch.utils.data import Dataset

class SingleFolderDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform = transform
        self.image_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]

        if len(self.image_paths) == 0:
            raise ValueError(f"No images found in the directory {folder_path}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        img_path = self.image_paths[index]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, 0  # Returning 0 as a dummy label


# Image dataset path
data_path = 'gan_data'

# Configure data loader
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

dataset = SingleFolderDataset(data_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# Move models to GPU
generator = generator.to(device)
discriminator = discriminator.to(device)

# Initialize variables to track the best losses
best_g_loss = float('inf')
best_d_loss = float('inf')

# Ensure the directory exists before saving images
os.makedirs('images', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)

# Wrap the outer loop with tqdm to show progress for epochs
for epoch in range(n_epochs):
    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    num_batches = len(dataloader)

    # Wrap the dataloader with tqdm to show progress for batches
    for i, (imgs, _) in enumerate(tqdm(dataloader, desc=f"Epoch {epoch}/{n_epochs}", leave=False)):

        # Move tensors to the configured device
        real_imgs = imgs.to(device)
        valid = torch.ones((imgs.size(0), 1), requires_grad=False).to(device)
        fake = torch.zeros((imgs.size(0), 1), requires_grad=False).to(device)
        z = torch.randn((imgs.size(0), latent_dim)).to(device)

        # -----------------
        #  Train Generator
        # -----------------

        optimizer_G.zero_grad()

        # Generate a batch of images
        gen_imgs = generator(z)

        # Loss measures generator's ability to fool the discriminator
        g_loss = adversarial_loss(discriminator(gen_imgs), valid)

        g_loss.backward()
        optimizer_G.step()

        # ---------------------
        #  Train Discriminator
        # ---------------------

        optimizer_D.zero_grad()

        # Measure discriminator's ability to classify real from generated samples
        real_loss = adversarial_loss(discriminator(real_imgs), valid)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach()), fake)
        d_loss = (real_loss + fake_loss) / 2

        d_loss.backward()
        optimizer_D.step()

        # Accumulate the epoch losses
        epoch_g_loss += g_loss.item()
        epoch_d_loss += d_loss.item()

        # Print progress and update tqdm bar
        if i % sample_interval == 0:
            tqdm.write(f"[Epoch {epoch}/{n_epochs}] [Batch {i}/{len(dataloader)}] [D loss: {d_loss.item()}] [G loss: {g_loss.item()}]")
            save_image(gen_imgs.data[:25].cpu(), f"images/{epoch}_{i}.png", nrow=5, normalize=True)

    # Calculate average losses for the epoch
    avg_g_loss = epoch_g_loss / num_batches
    avg_d_loss = epoch_d_loss / num_batches

    # Save the models with the least loss after the epoch
    if avg_g_loss < best_g_loss:
        best_g_loss = avg_g_loss
        torch.save(generator.state_dict(), "saved_models/best_generator.pth")
        tqdm.write(f"Saved new best generator model with average G loss: {best_g_loss}")

    if avg_d_loss < best_d_loss:
        best_d_loss = avg_d_loss
        torch.save(discriminator.state_dict(), "saved_models/best_discriminator.pth")
        tqdm.write(f"Saved new best discriminator model with average D loss: {best_d_loss}")


torch.save(generator.state_dict(), "generator.pth")
torch.save(discriminator.state_dict(), "discriminator.pth")


c:\Users\jesli\anaconda3\envs\pv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of parameters in the generator: 3448576
Number of parameters in the YOLO-like discriminator: 1570080
cuda


Epoch 0/250:   0%|          | 1/239 [00:25<1:40:56, 25.45s/it]

[Epoch 0/250] [Batch 0/239] [D loss: 0.6935117244720459] [G loss: 0.679198145866394]


Saved new best generator model with average G loss: 2.030494676474248
Saved new best discriminator model with average D loss: 0.1966973074933974


Epoch 1/250:   0%|          | 1/239 [00:01<04:35,  1.16s/it]

[Epoch 1/250] [Batch 0/239] [D loss: 0.059161316603422165] [G loss: 2.9124064445495605]


KeyboardInterrupt: 